In [9]:
# ============================================================
# ENERGY INTELLIGENCE HUB – EXPORT ALL REPORTS (FINAL)
# Includes: 7 Excel reports + 3 charts + Executive Summary
# ============================================================

import psycopg2
import pandas as pd
import os
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.drawing.image import Image
from datetime import datetime
import io

# ============================================================
# 1. CONNECTION
# ============================================================
conn = psycopg2.connect(
    host="localhost",
    database="energy_intelligence_project1",
    user="postgres",
    password="Solovigo123DELLYS"
)

# ============================================================
# 2. CREATE OUTPUTS FOLDER
# ============================================================
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../outputs/charts', exist_ok=True)

print("="*60)
print("GENERATING ENERGY REPORTS + CHARTS")
print("="*60)

# ============================================================
# 3. EXECUTIVE SUMMARY (Report 1)
# ============================================================
print("\nExecutive Summary...")

exec_summary = {
    'Metric': [
        'Total Buildings',
        'Total Energy Readings',
        'Total Anomalies Detected',
        'Average Load Factor (%)',
        'Highest Consumption Building',
        'Total Annual Bill (€)',
        'Forecast Next 48h (kWh)',
        'Report Generated'
    ],
    'Value': [
        50,
        '3,504,050',
        30,
        '~30%',
        'Building_31',
        '~€800,000',
        '12,450',
        datetime.now().strftime('%Y-%m-%d %H:%M')
    ]
}

df_exec_summary = pd.DataFrame(exec_summary)

# ============================================================
# 4. FORECAST CHART (Output 2)
# ============================================================
print("\nGenerating Forecast Chart...")

query_forecast = """
SELECT 
    timestamp AS ds,
    energy_kwh AS y
FROM energy_readings
WHERE building_id = 17
ORDER BY timestamp DESC
LIMIT 48;
"""
df_historical = pd.read_sql(query_forecast, conn)
df_historical = df_historical.sort_values('ds')

# Get forecast from energy_forecasts table
query_forecast_pred = """
SELECT 
    forecast_timestamp AS ds,
    forecast_energy_kwh AS yhat
FROM energy_forecasts
WHERE building_id = 17
ORDER BY forecast_timestamp
LIMIT 48;
"""
df_forecast = pd.read_sql(query_forecast_pred, conn)

# Plot
plt.figure(figsize=(12, 6))
plt.plot(df_historical['ds'], df_historical['y'], label='Actual (Last 48h)', color='blue', linewidth=2)
plt.plot(df_forecast['ds'], df_forecast['yhat'], label='Forecast (48h)', color='red', linestyle='--', linewidth=2)
plt.title('Building_17 - Actual vs Forecast (Next 48 Hours)', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Energy (kWh)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

forecast_chart_path = '../outputs/charts/forecast_chart.png'
plt.savefig(forecast_chart_path, dpi=150)
plt.close()
print(f"Forecast chart saved: {forecast_chart_path}")

# ============================================================
# 5. ANOMALY CHART (Output 3)
# ============================================================
print("\nGenerating Anomaly Chart...")

query_anomalies_chart = """
SELECT 
    er.timestamp,
    er.energy_kwh,
    CASE WHEN a.anomaly_id IS NOT NULL THEN 1 ELSE 0 END AS is_anomaly
FROM energy_readings er
LEFT JOIN anomalies a ON er.reading_id = a.reading_id
WHERE er.building_id = 17
ORDER BY er.timestamp DESC
LIMIT 1000;
"""
df_anomalies_chart = pd.read_sql(query_anomalies_chart, conn)
df_anomalies_chart = df_anomalies_chart.sort_values('timestamp')

# Separate normal and anomaly points
normal = df_anomalies_chart[df_anomalies_chart['is_anomaly'] == 0]
anomalies = df_anomalies_chart[df_anomalies_chart['is_anomaly'] == 1]

plt.figure(figsize=(12, 6))
plt.plot(normal['timestamp'], normal['energy_kwh'], label='Normal Consumption', color='blue', alpha=0.7, linewidth=1)
plt.scatter(anomalies['timestamp'], anomalies['energy_kwh'], color='red', s=50, label='Anomaly Detected', zorder=5)
plt.title('Building_17 - Anomaly Detection', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Energy (kWh)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

anomaly_chart_path = '../outputs/charts/anomaly_chart.png'
plt.savefig(anomaly_chart_path, dpi=150)
plt.close()
print(f"Anomaly chart saved: {anomaly_chart_path}")

# ============================================================
# 6. LOAD FACTOR RANKING (Report 4)
# ============================================================
print("\nLoad Factor Ranking...")

df_load_factor = pd.read_sql("""
SELECT 
    b.building_name,
    ROUND(SUM(er.energy_kwh), 2) AS total_kwh,
    ROUND(MAX(er.demand_kw), 2) AS peak_demand,
    ROUND(AVG(er.demand_kw), 2) AS avg_demand,
    ROUND((SUM(er.energy_kwh) / (MAX(er.demand_kw) * (COUNT(*) * 0.25))) * 100, 2) AS load_factor_percent,
    CASE 
        WHEN (SUM(er.energy_kwh) / (MAX(er.demand_kw) * (COUNT(*) * 0.25))) > 0.6 THEN 'Efficient'
        WHEN (SUM(er.energy_kwh) / (MAX(er.demand_kw) * (COUNT(*) * 0.25))) > 0.4 THEN 'Average'
        ELSE 'Inefficient'
    END AS efficiency_rating
FROM energy_readings er
JOIN buildings b ON er.building_id = b.building_id
GROUP BY b.building_name
ORDER BY load_factor_percent ASC;
""", conn)

# ============================================================
# 7. PEAK DEMAND (Report 5)
# ============================================================
print("\nPeak Demand (Top 10)...")

df_peak_demand = pd.read_sql("""
SELECT 
    b.building_name,
    pd.peak_date,
    pd.demand_kw,
    pd.demand_kw * 15.00 AS demand_charge_eur
FROM peak_demand pd
JOIN buildings b ON pd.building_id = b.building_id
ORDER BY demand_charge_eur DESC
LIMIT 10;
""", conn)

# ============================================================
# 8. ANOMALY SUMMARY (Report 6)
# ============================================================
print("\nAnomaly Summary...")

df_anomalies = pd.read_sql("""
SELECT 
    b.building_name,
    a.anomaly_type,
    a.severity,
    a.description,
    a.cost_impact_eur,
    a.status,
    a.detected_at
FROM anomalies a
JOIN buildings b ON a.building_id = b.building_id
ORDER BY a.cost_impact_eur DESC;
""", conn)

# ============================================================
# 9. ANNUAL DEMAND CHARGES (Report 7)
# ============================================================
print("\nAnnual Demand Charges...")

df_annual_demand = pd.read_sql("""
WITH monthly_peaks AS (
    SELECT 
        building_id,
        EXTRACT(YEAR FROM peak_date) AS year,
        DATE_TRUNC('month', peak_date) AS month,
        MAX(demand_kw) AS monthly_max_demand_kw
    FROM peak_demand
    GROUP BY building_id, EXTRACT(YEAR FROM peak_date), DATE_TRUNC('month', peak_date)
)
SELECT 
    b.building_name,
    mp.year,
    ROUND(SUM(mp.monthly_max_demand_kw * 15.00), 2) AS annual_demand_charge_eur
FROM monthly_peaks mp
JOIN buildings b ON mp.building_id = b.building_id
GROUP BY b.building_name, mp.year
ORDER BY annual_demand_charge_eur DESC;
""", conn)

# ============================================================
# 10. TOTAL ANNUAL BILL (Report 8)
# ============================================================
print("\nTotal Annual Bill...")

df_total_bill = pd.read_sql("""
WITH annual_energy AS (
    SELECT 
        b.building_id,
        b.building_name,
        EXTRACT(YEAR FROM er.timestamp) AS year,
        ROUND(SUM(er.energy_kwh) * 0.12, 2) AS energy_cost_eur
    FROM energy_readings er
    JOIN buildings b ON er.building_id = b.building_id
    GROUP BY b.building_id, b.building_name, EXTRACT(YEAR FROM er.timestamp)
),
monthly_peaks AS (
    SELECT 
        building_id,
        EXTRACT(YEAR FROM peak_date) AS year,
        DATE_TRUNC('month', peak_date) AS month,
        MAX(demand_kw) AS monthly_max_demand_kw
    FROM peak_demand
    GROUP BY building_id, EXTRACT(YEAR FROM peak_date), DATE_TRUNC('month', peak_date)
),
annual_demand AS (
    SELECT 
        building_id,
        year,
        ROUND(SUM(monthly_max_demand_kw * 15.00), 2) AS demand_charge_eur
    FROM monthly_peaks
    GROUP BY building_id, year
)
SELECT 
    ae.building_name,
    ae.year,
    ae.energy_cost_eur,
    ad.demand_charge_eur,
    912.50 AS fixed_fees_eur,
    ROUND(ae.energy_cost_eur + ad.demand_charge_eur + 912.50, 2) AS total_annual_bill_eur
FROM annual_energy ae
JOIN annual_demand ad ON ae.building_id = ad.building_id AND ae.year = ad.year
ORDER BY total_annual_bill_eur DESC;
""", conn)

# ============================================================
# 11. DEMAND CHARGE PERCENTAGE (Report 9)
# ============================================================
print("\nDemand Charge Percentage...")

df_demand_percentage = pd.read_sql("""
WITH annual_energy AS (
    SELECT 
        b.building_id,
        b.building_name,
        EXTRACT(YEAR FROM er.timestamp) AS year,
        ROUND(SUM(er.energy_kwh) * 0.12, 2) AS energy_cost_eur
    FROM energy_readings er
    JOIN buildings b ON er.building_id = b.building_id
    GROUP BY b.building_id, b.building_name, EXTRACT(YEAR FROM er.timestamp)
),
monthly_peaks AS (
    SELECT 
        building_id,
        EXTRACT(YEAR FROM peak_date) AS year,
        DATE_TRUNC('month', peak_date) AS month,
        MAX(demand_kw) AS monthly_max_demand_kw
    FROM peak_demand
    GROUP BY building_id, EXTRACT(YEAR FROM peak_date), DATE_TRUNC('month', peak_date)
),
annual_demand AS (
    SELECT 
        building_id,
        year,
        ROUND(SUM(monthly_max_demand_kw * 15.00), 2) AS demand_charge_eur
    FROM monthly_peaks
    GROUP BY building_id, year
)
SELECT 
    ae.building_name,
    ae.year,
    ae.energy_cost_eur,
    ad.demand_charge_eur,
    ROUND(ae.energy_cost_eur + ad.demand_charge_eur + 912.50, 2) AS total_bill_eur,
    ROUND((ad.demand_charge_eur / (ae.energy_cost_eur + ad.demand_charge_eur + 912.50)) * 100, 2) AS demand_percentage
FROM annual_energy ae
JOIN annual_demand ad ON ae.building_id = ad.building_id AND ae.year = ad.year
ORDER BY demand_percentage DESC;
""", conn)

# ============================================================
# 12. CLOSE CONNECTION
# ============================================================
conn.close()

# ============================================================
# 13. EXPORT ALL TO EXCEL (WITH EXECUTIVE SUMMARY FIRST)
# ============================================================
print("\nExporting all reports to Excel...")

output_path = '../outputs/energy_intelligence_reports.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # Executive Summary (FIRST SHEET)
    df_exec_summary.to_excel(writer, sheet_name='Executive Summary', index=False)
    print("Executive Summary")
    
    # All other reports
    reports = {
        'Load Factor Ranking': df_load_factor,
        'Peak Demand (Top 10)': df_peak_demand,
        'Anomaly Summary': df_anomalies,
        'Annual Demand Charges': df_annual_demand,
        'Total Annual Bill': df_total_bill,
        'Forecast Sample (48h)': df_forecast,
        'Demand Charge %': df_demand_percentage
    }
    
    for sheet_name, df in reports.items():
        if not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"{sheet_name}: {len(df)} rows")
        else:
            print(f"{sheet_name}: EMPTY (skipped)")

print(f"\nAll reports saved to: {output_path}")
print("File location: outputs/energy_intelligence_reports.xlsx")

# ============================================================
# 14. CHARTS SUMMARY
# ============================================================
print("\n" + "="*60)
print("ALL REPORTS + CHARTS GENERATED SUCCESSFULLY!")
print("="*60)
print("Reports included:")
print("Executive Summary (NEW)")
print("Load Factor Ranking")
print("Peak Demand (Top 10)")
print("Anomaly Summary")
print("Annual Demand Charges")
print("Total Annual Bill")
print("Forecast Sample (48h)")
print("Demand Charge %")
print("\nCharts included (outputs/charts/):")
print("forecast_chart.png")
print("anomaly_chart.png")

GENERATING ENERGY REPORTS + CHARTS

Executive Summary...

Generating Forecast Chart...


C:\Users\DeyStore\AppData\Local\Temp\ipykernel_14992\11034342.py:79: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_historical = pd.read_sql(query_forecast, conn)
C:\Users\DeyStore\AppData\Local\Temp\ipykernel_14992\11034342.py:92: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_forecast = pd.read_sql(query_forecast_pred, conn)


Forecast chart saved: ../outputs/charts/forecast_chart.png

Generating Anomaly Chart...


C:\Users\DeyStore\AppData\Local\Temp\ipykernel_14992\11034342.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_anomalies_chart = pd.read_sql(query_anomalies_chart, conn)


Anomaly chart saved: ../outputs/charts/anomaly_chart.png

Load Factor Ranking...


...


Anomaly Summary...

Annual Demand Charges...


C:\Users\DeyStore\AppData\Local\Temp\ipykernel_14992\11034342.py:195: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_anomalies = pd.read_sql("""
C:\Users\DeyStore\AppData\Local\Temp\ipykernel_14992\11034342.py:214: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_annual_demand = pd.read_sql("""



Total Annual Bill...


C:\Users\DeyStore\AppData\Local\Temp\ipykernel_14992\11034342.py:239: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_total_bill = pd.read_sql("""


KeyboardInterrupt: 